# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import collections

# Import DataFrame

In [2]:
sampled_df_filepath = "sampled_dataframe.pkl"

sampled_df = pd.read_pickle(sampled_df_filepath).reset_index(drop=True)

### Getting Unique Vocabulary

In [3]:
vocab = set()

for sublist in sampled_df['lemmas']:
    vocab.update(sublist)

vocab_size = len(vocab)

print(f"Vocab size is {vocab_size} lemmas.")

Vocab size is 159862 lemmas.


### Creating Dictionary

In [4]:
word_to_id = {word: idx for idx, word in enumerate(vocab)}

#### Generating BoW Representations

<div class="alert alert-block alert-danger">
  <b>Danger:</b> Do not run the following on the entire dataset.
</div>

In [ ]:
bow_representations = np.zeros((sampled_df.shape[0], vocab_size), dtype=np.uint16)

for (idx, sentence) in enumerate(sampled_df['lemmas']):
    for lemma in sentence:
        bow_representations[idx, word_to_id[lemma]] += 1

# 1D Convolutions

The following cell determines the minimum vocabulary size needed to cover 80% of all token occurrences in the corpus by calculating cumulative relative word frequencies.

In [ ]:
token_frequencies = collections.Counter()

for sentence in sampled_df["lemmas"]:
    token_frequencies.update(sentence)

sorted_counts = np.sort(np.fromiter(token_frequencies.values(), dtype=np.uint32), descending=True)
cumulative_sorted_counts = np.cumsum(sorted_counts)
normalized_cumulative_sorted_counts = cumulative_sorted_counts / cumulative_sorted_counts[-1]

FREQUENCY_THRESHOLD = 0.8
threshold = np.searchsorted(normalized_cumulative_sorted_counts, FREQUENCY_THRESHOLD) + 1

print(f"We will use the most frequent {threshold} lemmas as our vocab and insert an additional unknown character (<UNK>) for everything else.")